# Supply Chain Control Tower – Risk Monitoring Layer

## Objective

This notebook develops the operational risk monitoring layer for the Supply Chain Control Tower.

The objective is to transform transactional demand data into actionable supply chain risk indicators that support capacity planning, bottleneck identification, inventory monitoring, and operational decision-making.

## Scope

Current Risk Modules:

- Capacity Risk

Future Risk Modules:

- Demand Risk
- Inventory Risk
- Service Risk
- Profitability Risk

## Business Questions

This notebook aims to answer the following questions:

1. Which manufacturing plants are approaching capacity constraints?

2. Where are potential bottlenecks occurring?

3. What products are driving capacity utilization increases?

4. Which operational risks require management attention?

## Data Sources

Master Data:
- dim_product
- dim_plant
- product_plant_mapping
- capacity_table

Processed Data:
- weekly_plant_demand

## Deliverables

- Capacity Utilization KPI
- Available Capacity KPI
- Capacity Alert Logic
- Bottleneck Monitoring Dataset
- Root Cause Analysis Dataset

The outputs generated in this notebook will serve as the analytical foundation for the Supply Chain Control Tower Dashboard.

### ==========================================
### Load Data
### ==========================================

In [1]:
import pandas as pd
import numpy as np

In [2]:
weekly_plant_demand = pd.read_csv(
    "../data/processed/weekly_plant_demand.csv"
)

capacity_table = pd.read_csv(
    "../data/master/capacity_table.csv"
)

df = pd.read_csv(
    "../data/raw/DataCoSupplyChainDataset.csv",
    encoding="latin1"
)

product_plant_mapping = pd.read_csv(
    "../data/master/product_plant_mapping.csv"
)

In [3]:
capacity_utilization = (
    weekly_plant_demand.merge(
        capacity_table[
            ["Plant_ID", "Weekly_Capacity"]
        ],
        on="Plant_ID",
        how="left"
    )
)

capacity_utilization.head()

,Plant_ID,Order_Date,Order Item Quantity,Weekly_Capacity
0,P1,2015-01-04,567,1165
1,P1,2015-01-11,969,1165
2,P1,2015-01-18,988,1165
3,P1,2015-01-25,1009,1165
4,P1,2015-02-01,963,1165


In [4]:
# Total demand assigned to a plant in a given week

capacity_utilization = capacity_utilization.rename(
    columns={
        "Order Item Quantity": "Weekly_Demand"
    }
)

In [5]:
# Percentage of plant capacity consumed by weekly demand

capacity_utilization["Utilization"] = (
    capacity_utilization["Weekly_Demand"]
    /
    capacity_utilization["Weekly_Capacity"]
)

In [6]:
# Remaining production capacity available after fulfilling weekly demand

capacity_utilization["Available_Capacity"] = (
    capacity_utilization["Weekly_Capacity"]
    -
    capacity_utilization["Weekly_Demand"]
)

In [7]:
# Capacity risk level based on utilization threshold
def capacity_status(util):

    if util >= 0.95:
        return "Critical"

    elif util >= 0.85:
        return "Warning"

    else:
        return "Normal"

capacity_utilization["Capacity_Status"] = (
    capacity_utilization["Utilization"]
    .apply(capacity_status)
)

In [8]:
capacity_utilization.head()
capacity_utilization

,Plant_ID,Order_Date,Weekly_Demand,Weekly_Capacity,Utilization,Available_Capacity,Capacity_Status
0,P1,2015-01-04,567,1165,0.486695,598,Normal
1,P1,2015-01-11,969,1165,0.831760,196,Normal
2,P1,2015-01-18,988,1165,0.848069,177,Normal
3,P1,2015-01-25,1009,1165,0.866094,156,Warning
4,P1,2015-02-01,963,1165,0.826609,202,Normal
...,...,...,...,...,...,...,...
481,P3,2018-01-07,352,913,0.385542,561,Normal
482,P3,2018-01-14,269,913,0.294633,644,Normal
483,P3,2018-01-21,479,913,0.524644,434,Normal
484,P3,2018-01-28,304,913,0.332968,609,Normal


In [9]:
capacity_utilization

,Plant_ID,Order_Date,Weekly_Demand,Weekly_Capacity,Utilization,Available_Capacity,Capacity_Status
0,P1,2015-01-04,567,1165,0.486695,598,Normal
1,P1,2015-01-11,969,1165,0.831760,196,Normal
2,P1,2015-01-18,988,1165,0.848069,177,Normal
3,P1,2015-01-25,1009,1165,0.866094,156,Warning
4,P1,2015-02-01,963,1165,0.826609,202,Normal
...,...,...,...,...,...,...,...
481,P3,2018-01-07,352,913,0.385542,561,Normal
482,P3,2018-01-14,269,913,0.294633,644,Normal
483,P3,2018-01-21,479,913,0.524644,434,Normal
484,P3,2018-01-28,304,913,0.332968,609,Normal


In [10]:
capacity_utilization.to_csv(
    "../data/processed/capacity_utilization_weekly.csv",
    index=False
)

### ==========================================
### Product Root Cause Analysis Dataset
### ==========================================

In [11]:
plant_product_weekly_demand = (
    df.merge(
        product_plant_mapping[
            [
                "Product Card Id",
                "Plant_ID"
            ]
        ],
        on="Product Card Id",
        how="left"
    )
)

In [12]:
plant_product_weekly_demand = (
    plant_product_weekly_demand[
        [
            "order date (DateOrders)",
            "Plant_ID",
            "Product Card Id",
            "Product Name",
            "Order Item Quantity"
        ]
    ]
)

In [13]:
# Order week used for product-level demand monitoring

plant_product_weekly_demand[
    "Order_Date"
] = pd.to_datetime(
    plant_product_weekly_demand[
        "order date (DateOrders)"
    ]
)

In [14]:
# Weekly demand by product within each plant

plant_product_weekly_demand = (
    plant_product_weekly_demand
    .set_index("Order_Date")
    .groupby(
        [
            "Plant_ID",
            "Product Card Id",
            "Product Name"
        ]
    )
    ["Order Item Quantity"]
    .resample("W")
    .sum()
    .reset_index()
)

In [15]:
# Total product demand assigned to a plant during a week

plant_product_weekly_demand = (
    plant_product_weekly_demand.rename(
        columns={
            "Order Item Quantity": "Weekly_Demand"
        }
    )
)

In [16]:
plant_product_weekly_demand.head()

,Plant_ID,Product Card Id,Product Name,Order_Date,Weekly_Demand
0,P1,172,Nike Women's Tempo Shorts,2015-01-04,6
1,P1,172,Nike Women's Tempo Shorts,2015-01-11,11
2,P1,172,Nike Women's Tempo Shorts,2015-01-18,11
3,P1,172,Nike Women's Tempo Shorts,2015-01-25,18
4,P1,172,Nike Women's Tempo Shorts,2015-02-01,4


In [17]:
'''
plant_product_weekly_demand.to_csv(
    "../data/processed/plant_product_weekly_demand.csv",
    index=False
)
'''

'\nplant_product_weekly_demand.to_csv(\n    "../data/processed/plant_product_weekly_demand.csv",\n    index=False\n)\n'

### ==========================================
### Top Demand Product Driver
### ==========================================

In [36]:
plant_product_weekly_demand = pd.read_csv(r'../data/processed/plant_product_weekly_demand.csv')

In [37]:
# Product demand ranking within each plant and week

top_product_driver = (
    plant_product_weekly_demand
    .sort_values(
        [
            "Plant_ID",
            "Order_Date",
            "Weekly_Demand"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

In [38]:
# Demand contribution rank within a plant-week

top_product_driver["Demand_Rank"] = (
    top_product_driver
    .groupby(
        [
            "Plant_ID",
            "Order_Date"
        ]
    )["Weekly_Demand"]
    .rank(
        method="dense",
        ascending=False
    )
)

### ==========================================
### Weekly Demand Change
### ==========================================

In [39]:
# Ensure product demand history is ordered chronologically

plant_product_weekly_demand = (
    plant_product_weekly_demand
    .sort_values(
        [
            "Plant_ID",
            "Product Card Id",
            "Order_Date"
        ]
    )
)

In [40]:
# Product demand observed in the previous week

plant_product_weekly_demand["Demand_Last_Week"] = (
    plant_product_weekly_demand
    .groupby(
        [
            "Plant_ID",
            "Product Card Id"
        ]
    )["Weekly_Demand"]
    .shift(1)
)

In [41]:
# Absolute increase in demand compared with previous week

plant_product_weekly_demand["Demand_Change"] = (
    plant_product_weekly_demand["Weekly_Demand"]
    -
    plant_product_weekly_demand["Demand_Last_Week"]
)

In [42]:
# Week-over-week demand growth percentage

plant_product_weekly_demand["Demand_Growth"] = (
    plant_product_weekly_demand["Demand_Change"]
    /
    plant_product_weekly_demand["Demand_Last_Week"]
)

In [43]:
plant_product_weekly_demand = (
    plant_product_weekly_demand
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

In [44]:
plant_product_weekly_demand[
    [
        "Plant_ID",
        "Product Name",
        "Order_Date",
        "Weekly_Demand",
        "Demand_Last_Week",
        "Demand_Change",
        "Demand_Growth"
    ]
].head(20)

,Plant_ID,Product Name,Order_Date,Weekly_Demand,Demand_Last_Week,Demand_Change,Demand_Growth
0,P1,Nike Women's Tempo Shorts,2015-01-04,6,NaN,NaN,NaN
1,P1,Nike Women's Tempo Shorts,2015-01-11,11,6.0,5.0,0.833333
2,P1,Nike Women's Tempo Shorts,2015-01-18,11,11.0,0.0,0.000000
3,P1,Nike Women's Tempo Shorts,2015-01-25,18,11.0,7.0,0.636364
4,P1,Nike Women's Tempo Shorts,2015-02-01,4,18.0,-14.0,-0.777778
5,P1,Nike Women's Tempo Shorts,2015-02-08,9,4.0,5.0,1.250000
6,P1,Nike Women's Tempo Shorts,2015-02-15,9,9.0,0.0,0.000000
7,P1,Nike Women's Tempo Shorts,2015-02-22,9,9.0,0.0,0.000000
8,P1,Nike Women's Tempo Shorts,2015-03-01,5,9.0,-4.0,-0.444444
9,P1,Nike Women's Tempo Shorts,2015-03-08,6,5.0,1.0,0.200000


In [45]:
top_product_driver = (
    top_product_driver.merge(
        plant_product_weekly_demand[
            [
                "Plant_ID",
                "Product Card Id",
                "Order_Date",
                "Demand_Last_Week",
                "Demand_Change",
                "Demand_Growth"
            ]
        ],
        on=[
            "Plant_ID",
            "Product Card Id",
            "Order_Date"
        ],
        how="left"
    )
)

In [46]:
# Top 10 demand-contributing products within a plant-week

top_product_driver = (
    top_product_driver[
        top_product_driver["Demand_Rank"] <= 10
    ]
)

In [47]:
top_product_driver.head(20)


,Plant_ID,Product Card Id,Product Name,Order_Date,Weekly_Demand,Demand_Rank,Demand_Last_Week,Demand_Change,Demand_Growth
0,P1,365,Perfect Fitness Perfect Rip Deck,2015-01-04,306,1.0,NaN,NaN,NaN
1,P1,191,Nike Men's Free 5.0+ Running Shoe,2015-01-04,160,2.0,NaN,NaN,NaN
2,P1,403,Nike Men's CJ Elite 2 TD Football Cleat,2015-01-04,77,3.0,NaN,NaN,NaN
3,P1,276,Under Armour Women's Ignite Slide,2015-01-04,10,4.0,NaN,NaN,NaN
4,P1,172,Nike Women's Tempo Shorts,2015-01-04,6,5.0,NaN,NaN,NaN
5,P1,235,Under Armour Hustle Storm Medium Duffle Bag,2015-01-04,4,6.0,NaN,NaN,NaN
6,P1,278,Under Armour Men's Compression EV SL Slide,2015-01-04,4,6.0,NaN,NaN,NaN
7,P1,365,Perfect Fitness Perfect Rip Deck,2015-01-11,529,1.0,306.0,223.0,0.728758
8,P1,191,Nike Men's Free 5.0+ Running Shoe,2015-01-11,243,2.0,160.0,83.0,0.518750
9,P1,403,Nike Men's CJ Elite 2 TD Football Cleat,2015-01-11,147,3.0,77.0,70.0,0.909091


In [30]:
top_product_driver.groupby(
    ["Plant_ID", "Order_Date"]
).size().describe()

count    481.000000
mean       9.852391
std        5.718486
min        1.000000
25%        5.000000
50%       10.000000
75%       13.000000
max       31.000000
dtype: float64

### ==========================================
### Demand Growth Driver
### ==========================================

In [50]:
# Total plant demand during a given week

plant_weekly_total_demand = (
    plant_product_weekly_demand
    .groupby(
        [
            "Plant_ID",
            "Order_Date"
        ]
    )["Weekly_Demand"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "Weekly_Demand": "Plant_Weekly_Demand"
        }
    )
)

In [51]:
top_product_driver = (
    top_product_driver.merge(
        plant_weekly_total_demand,
        on=[
            "Plant_ID",
            "Order_Date"
        ],
        how="left"
    )
)

In [52]:
# Percentage contribution to total plant demand

top_product_driver["Demand_Contribution"] = (
    top_product_driver["Weekly_Demand"]
    /
    top_product_driver["Plant_Weekly_Demand"]
)

In [53]:
top_product_driver.head()

,Plant_ID,Product Card Id,Product Name,Order_Date,Weekly_Demand,Demand_Rank,Demand_Last_Week,Demand_Change,Demand_Growth,Plant_Weekly_Demand,Demand_Contribution
0,P1,365,Perfect Fitness Perfect Rip Deck,2015-01-04,306,1.0,NaN,NaN,NaN,567,0.539683
1,P1,191,Nike Men's Free 5.0+ Running Shoe,2015-01-04,160,2.0,NaN,NaN,NaN,567,0.282187
2,P1,403,Nike Men's CJ Elite 2 TD Football Cleat,2015-01-04,77,3.0,NaN,NaN,NaN,567,0.135802
3,P1,276,Under Armour Women's Ignite Slide,2015-01-04,10,4.0,NaN,NaN,NaN,567,0.017637
4,P1,172,Nike Women's Tempo Shorts,2015-01-04,6,5.0,NaN,NaN,NaN,567,0.010582


In [54]:
# Percentage contribution to total plant demand

top_product_driver["Demand_Contribution_Pct"] = (
    top_product_driver["Demand_Contribution"]
    * 100
).round(2)

In [ ]:
# Cumulative demand contribution within a plant-week
top_product_driver = (
    top_product_driver
    .sort_values(
        [
            "Plant_ID",
            "Order_Date",
            "Demand_Rank"
        ]
    )
)



top_product_driver["Cumulative_Contribution"] = (
    top_product_driver
    .groupby(
        [
            "Plant_ID",
            "Order_Date"
        ]
    )["Demand_Contribution"]
    .cumsum()
)

In [56]:
top_product_driver.columns

Index(['Plant_ID', 'Product Card Id', 'Product Name', 'Order_Date',
       'Weekly_Demand', 'Demand_Rank', 'Demand_Last_Week', 'Demand_Change',
       'Demand_Growth', 'Plant_Weekly_Demand', 'Demand_Contribution',
       'Demand_Contribution_Pct', 'Cumulative_Contribution'],
      dtype='object')

In [57]:
top_product_driver.head(20)

,Plant_ID,Product Card Id,Product Name,Order_Date,Weekly_Demand,Demand_Rank,Demand_Last_Week,Demand_Change,Demand_Growth,Plant_Weekly_Demand,Demand_Contribution,Demand_Contribution_Pct,Cumulative_Contribution
0,P1,365,Perfect Fitness Perfect Rip Deck,2015-01-04,306,1.0,NaN,NaN,NaN,567,0.539683,53.97,0.539683
1,P1,191,Nike Men's Free 5.0+ Running Shoe,2015-01-04,160,2.0,NaN,NaN,NaN,567,0.282187,28.22,0.821869
2,P1,403,Nike Men's CJ Elite 2 TD Football Cleat,2015-01-04,77,3.0,NaN,NaN,NaN,567,0.135802,13.58,0.957672
3,P1,276,Under Armour Women's Ignite Slide,2015-01-04,10,4.0,NaN,NaN,NaN,567,0.017637,1.76,0.975309
4,P1,172,Nike Women's Tempo Shorts,2015-01-04,6,5.0,NaN,NaN,NaN,567,0.010582,1.06,0.985891
5,P1,235,Under Armour Hustle Storm Medium Duffle Bag,2015-01-04,4,6.0,NaN,NaN,NaN,567,0.007055,0.71,0.992945
6,P1,278,Under Armour Men's Compression EV SL Slide,2015-01-04,4,6.0,NaN,NaN,NaN,567,0.007055,0.71,1.000000
7,P1,365,Perfect Fitness Perfect Rip Deck,2015-01-11,529,1.0,306.0,223.0,0.728758,969,0.545924,54.59,0.545924
8,P1,191,Nike Men's Free 5.0+ Running Shoe,2015-01-11,243,2.0,160.0,83.0,0.518750,969,0.250774,25.08,0.796698
9,P1,403,Nike Men's CJ Elite 2 TD Football Cleat,2015-01-11,147,3.0,77.0,70.0,0.909091,969,0.151703,15.17,0.948400


In [ ]:
"""
top_product_driver.to_csv(
    "../data/processed/top_product_driver_weekly.csv",
    index=False
)
"""

# Product Inventory Baseline

## Objective

The purpose of this step is to establish a starting inventory position for each product in the absence of actual inventory records.

Since the DataCo dataset does not contain inventory, replenishment, or production data, inventory levels are estimated using demand-based inventory policies derived from ABC inventory classification.

## Business Logic

Inventory policies are assigned based on product importance:

| ABC Class | Safety Stock Days | Target Coverage Days |
|------------|------------------:|---------------------:|
| A | 21 | 45 |
| B | 14 | 30 |
| C | 7 | 21 |

The initial inventory level is estimated using the target coverage policy.

## Calculation

Average Daily Demand

= Average Weekly Demand / 7

Initial Inventory

= Average Daily Demand × Target Coverage Days

## Deliverable

product_inventory_baseline.csv

Key Columns:

- Product Card Id
- Product Name
- ABC_Class
- Average_Weekly_Demand
- Average_Daily_Demand
- Safety_Stock_Days
- Target_Coverage_Days
- Initial_Inventory

## Business Purpose

This baseline inventory serves as the starting point for inventory simulation and inventory risk monitoring.

The resulting inventory levels will be used to calculate:

- Inventory On Hand
- Days of Inventory (DOI)
- Inventory Status
- Reorder Risk
- Excess Inventory Risk

## Product ABC Classification

### Objective

Classify products based on historical demand contribution using ABC inventory classification.

ABC classification will be used to derive inventory policies and inventory risk thresholds.

In [74]:
plant_product_weekly_demand = pd.read_csv(r'../data/processed/plant_product_weekly_demand.csv')

In [75]:
# ==========================================
# Product ABC Classification
# ==========================================

product_abc = (
    plant_product_weekly_demand
    .groupby(
        [
            "Product Card Id",
            "Product Name"
        ]
    )["Weekly_Demand"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "Weekly_Demand": "Total_Demand"
        }
    )
)

product_abc.head()

,Product Card Id,Product Name,Total_Demand
0,19,Nike Men's Fingertrap Max Training Shoe,64
1,24,Elevation Training Mask 2.0,231
2,35,adidas Brazuca 2014 Official Match Ball,65
3,37,adidas Kids' F5 Messi FG Soccer Cleat,781
4,44,adidas Men's F10 Messi TRX FG Soccer Cleat,939


In [76]:
# Rank products by total historical demand

product_abc = (
    product_abc
    .sort_values(
        "Total_Demand",
        ascending=False
    )
    .reset_index(drop=True)
)

In [77]:
# Percentage contribution to total demand

total_demand = product_abc["Total_Demand"].sum()

product_abc["Demand_Share"] = (
    product_abc["Total_Demand"]
    /
    total_demand
)
product_abc.head()

,Product Card Id,Product Name,Total_Demand,Demand_Share
0,365,Perfect Fitness Perfect Rip Deck,73698,0.191882
1,502,Nike Men's Dri-FIT Victory Golf Polo,62956,0.163914
2,1014,O'Brien Men's Neoprene Life Vest,57803,0.150498
3,191,Nike Men's Free 5.0+ Running Shoe,36680,0.095501
4,627,Under Armour Girls' Toddler Spine Surge Runni,31735,0.082626


In [78]:
# Cumulative demand contribution

product_abc["Cumulative_Demand_Share"] = (
    product_abc["Demand_Share"]
    .cumsum()
)

product_abc.head()

,Product Card Id,Product Name,Total_Demand,Demand_Share,Cumulative_Demand_Share
0,365,Perfect Fitness Perfect Rip Deck,73698,0.191882,0.191882
1,502,Nike Men's Dri-FIT Victory Golf Polo,62956,0.163914,0.355797
2,1014,O'Brien Men's Neoprene Life Vest,57803,0.150498,0.506294
3,191,Nike Men's Free 5.0+ Running Shoe,36680,0.095501,0.601795
4,627,Under Armour Girls' Toddler Spine Surge Runni,31735,0.082626,0.684422


In [79]:
# ABC inventory classification based on cumulative demand contribution

def assign_abc_class(cum_share):

    if cum_share <= 0.70:
        return "A"

    elif cum_share <= 0.90:
        return "B"

    else:
        return "C"
    

product_abc["ABC_Class"] = (
    product_abc["Cumulative_Demand_Share"]
    .apply(assign_abc_class)
)
product_abc.head(30)

,Product Card Id,Product Name,Total_Demand,Demand_Share,Cumulative_Demand_Share,ABC_Class
0,365,Perfect Fitness Perfect Rip Deck,73698,0.191882,0.191882,A
1,502,Nike Men's Dri-FIT Victory Golf Polo,62956,0.163914,0.355797,A
2,1014,O'Brien Men's Neoprene Life Vest,57803,0.150498,0.506294,A
3,191,Nike Men's Free 5.0+ Running Shoe,36680,0.095501,0.601795,A
4,627,Under Armour Girls' Toddler Spine Surge Runni,31735,0.082626,0.684422,A
5,403,Nike Men's CJ Elite 2 TD Football Cleat,22246,0.057920,0.742342,B
6,1004,Field & Stream Sportsman 16 Gun Fire Safe,17325,0.045108,0.787450,B
7,1073,Pelican Sunstream 100 Kayak,15500,0.040356,0.827806,B
8,957,Diamondback Women's Serene Classic Comfort Bi,13729,0.035745,0.863552,B
9,977,ENO Atlas Hammock Straps,998,0.002598,0.866150,B


In [80]:
product_abc["ABC_Class"].value_counts()

ABC_Class
C    95
B    18
A     5
Name: count, dtype: int64

In [81]:
product_abc.groupby(
    "ABC_Class"
)["Total_Demand"].sum()

ABC_Class
A    262872
B     82000
C     39207
Name: Total_Demand, dtype: int64

In [82]:
product_abc.groupby(
    "ABC_Class"
)["Demand_Share"].sum()

ABC_Class
A    0.684422
B    0.213498
C    0.102081
Name: Demand_Share, dtype: float64

In [ ]:
#product_abc.to_csv(
#    "../data/processed/product_abc_classification.csv",
#    index=False)

### ==========================================
### Inventory Policy Mapping
### ==========================================

In [109]:
inventory_policy_mapping = {
    "A": {
        "Safety_Stock_Days": 21,
        "Target_Coverage_Days": 45
    },
    "B": {
        "Safety_Stock_Days": 14,
        "Target_Coverage_Days": 30
    },
    "C": {
        "Safety_Stock_Days": 7,
        "Target_Coverage_Days": 21
    }
}

In [110]:
# Inventory policy derived from ABC classification

inventory_policy = (
    product_abc[
        [
            "Product Card Id",
            "Product Name",
            "ABC_Class"
        ]
    ]
    .copy()
)

In [111]:
# Minimum inventory coverage requirement

inventory_policy["Safety_Stock_Days"] = (
    inventory_policy["ABC_Class"]
    .map(
        {
            k: v["Safety_Stock_Days"]
            for k, v in inventory_policy_mapping.items()
        }
    )
)

In [112]:
inventory_policy.head()

,Product Card Id,Product Name,ABC_Class,Safety_Stock_Days
0,365,Perfect Fitness Perfect Rip Deck,A,21
1,502,Nike Men's Dri-FIT Victory Golf Polo,A,21
2,1014,O'Brien Men's Neoprene Life Vest,A,21
3,191,Nike Men's Free 5.0+ Running Shoe,A,21
4,627,Under Armour Girls' Toddler Spine Surge Runni,A,21


In [113]:
# Target inventory coverage level

inventory_policy["Target_Coverage_Days"] = (
    inventory_policy["ABC_Class"]
    .map(
        {
            k: v["Target_Coverage_Days"]
            for k, v in inventory_policy_mapping.items()
        }
    )
)
inventory_policy.head()

,Product Card Id,Product Name,ABC_Class,Safety_Stock_Days,Target_Coverage_Days
0,365,Perfect Fitness Perfect Rip Deck,A,21,45
1,502,Nike Men's Dri-FIT Victory Golf Polo,A,21,45
2,1014,O'Brien Men's Neoprene Life Vest,A,21,45
3,191,Nike Men's Free 5.0+ Running Shoe,A,21,45
4,627,Under Armour Girls' Toddler Spine Surge Runni,A,21,45


In [114]:
inventory_policy.groupby(
    "ABC_Class"
)[
    [
        "Safety_Stock_Days",
        "Target_Coverage_Days"
    ]
].first()

,Safety_Stock_Days,Target_Coverage_Days
ABC_Class,,
A,21,45
B,14,30
C,7,21


In [ ]:
#inventory_policy.to_csv(
#    "../data/master/inventory_policy.csv",
#    index=False)

### ==========================================
### Product Inventory Baseline
### ==========================================

In [116]:
product_inventory_baseline = (
    product_abc.merge(
        inventory_policy[
            [
                "Product Card Id",
                "Safety_Stock_Days",
                "Target_Coverage_Days"
            ]
        ],
        on="Product Card Id",
        how="left"
    )
)

In [117]:
# Average weekly demand per product

avg_weekly_demand = (
    plant_product_weekly_demand
    .groupby(
        [
            "Product Card Id"
        ]
    )["Weekly_Demand"]
    .mean()
    .reset_index()
    .rename(
        columns={
            "Weekly_Demand":"Average_Weekly_Demand"
        }
    )
)

In [118]:
product_inventory_baseline = (
    product_inventory_baseline.merge(
        avg_weekly_demand,
        on="Product Card Id",
        how="left"
    )
)

In [119]:
# Average daily demand

product_inventory_baseline["Average_Daily_Demand"] = (
    product_inventory_baseline["Average_Weekly_Demand"]
    / 7
)

In [120]:
# Starting inventory level based on target coverage policy
inventory_buffer = {
    "A": 2.0,
    "B": 3.0,
    "C": 5.0
}

product_inventory_baseline["Buffer_Factor"] = (
    product_inventory_baseline["ABC_Class"]
    .map(inventory_buffer)
)

product_inventory_baseline["Initial_Inventory"] = (
    product_inventory_baseline["Average_Daily_Demand"]
    *
    product_inventory_baseline["Target_Coverage_Days"]
    *
    product_inventory_baseline["Buffer_Factor"]
).round().astype(int)

In [121]:
product_inventory_baseline.head()

,Product Card Id,Product Name,Total_Demand,Demand_Share,Cumulative_Demand_Share,ABC_Class,Safety_Stock_Days,Target_Coverage_Days,Average_Weekly_Demand,Average_Daily_Demand,Buffer_Factor,Initial_Inventory
0,365,Perfect Fitness Perfect Rip Deck,73698,0.191882,0.191882,A,21,45,508.262069,72.608867,2.0,6535
1,502,Nike Men's Dri-FIT Victory Golf Polo,62956,0.163914,0.355797,A,21,45,434.179310,62.025616,2.0,5582
2,1014,O'Brien Men's Neoprene Life Vest,57803,0.150498,0.506294,A,21,45,398.641379,56.948768,2.0,5125
3,191,Nike Men's Free 5.0+ Running Shoe,36680,0.095501,0.601795,A,21,45,252.965517,36.137931,2.0,3252
4,627,Under Armour Girls' Toddler Spine Surge Runni,31735,0.082626,0.684422,A,21,45,218.862069,31.266010,2.0,2814


In [ ]:
#product_inventory_baseline.to_csv(
#    "../data/processed/product_inventory_baseline.csv",
#    index=False)

# Inventory Snapshot Simulation

## Objective

Simulate weekly inventory positions using demand consumption and periodic replenishment policies.

Since the DataCo dataset does not contain actual inventory transactions, a simplified inventory simulation is used to estimate inventory positions over time.

## Inventory Simulation Logic

1. Initial inventory is established using inventory policies derived from ABC classification.

2. Weekly demand consumes inventory.

3. Inventory is reviewed every 4 weeks.

4. During each review cycle, inventory is replenished back to the target coverage level.

## Inventory Flow

Beginning Inventory  
↓  
Weekly Demand Consumption  
↓  
Ending Inventory  
↓  
Periodic Replenishment (Every 4 Weeks)  
↓  
Next Week Beginning Inventory

## Deliverable

inventory_snapshot_weekly.csv

Key Columns:

- Product Card Id
- Product Name
- Order_Date
- Weekly_Demand
- Beginning_Inventory
- Ending_Inventory
- Replenishment_Qty

### ==========================================
### Inventory Snapshot Simulation
### ==========================================

In [123]:
inventory_snapshot = (
    plant_product_weekly_demand.merge(
        product_inventory_baseline[
            [
                "Product Card Id",
                "Product Name",
                "Initial_Inventory",
                "Target_Coverage_Days",
                "Average_Daily_Demand"
            ]
        ],
        on=[
            "Product Card Id",
            "Product Name"
        ],
        how="left"
    )
)

In [124]:
# Ensure inventory simulation follows chronological order

inventory_snapshot = (
    inventory_snapshot
    .sort_values(
        [
            "Product Card Id",
            "Order_Date"
        ]
    )
    .reset_index(drop=True)
)

In [125]:
# Inventory review cycle identifier

inventory_snapshot["Review_Cycle"] = (
    inventory_snapshot
    .groupby("Product Card Id")
    .cumcount()
    // 4
)

In [126]:
inventory_snapshot.head()

,Plant_ID,Product Card Id,Product Name,Order_Date,Weekly_Demand,Initial_Inventory,Target_Coverage_Days,Average_Daily_Demand,Review_Cycle
0,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-04-30,2,48,21,0.457143,0
1,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-07,3,48,21,0.457143,0
2,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-14,1,48,21,0.457143,0
3,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-21,3,48,21,0.457143,0
4,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-28,6,48,21,0.457143,1


In [127]:
# Inventory simulation metrics

inventory_snapshot["Beginning_Inventory"] = np.nan

inventory_snapshot["Ending_Inventory"] = np.nan

inventory_snapshot["Replenishment_Qty"] = 0

In [128]:
# Target inventory level based on coverage policy

inventory_snapshot["Target_Inventory"] = (
    inventory_snapshot["Average_Daily_Demand"]
    *
    inventory_snapshot["Target_Coverage_Days"]
).round().astype(int)

In [129]:
for product_id in inventory_snapshot["Product Card Id"].unique():

    product_idx = (
        inventory_snapshot[
            inventory_snapshot["Product Card Id"]
            == product_id
        ]
        .index
        .tolist()
    )

    current_inventory = int(
        inventory_snapshot.loc[
            product_idx[0],
            "Initial_Inventory"
        ]
    )

    for i, idx in enumerate(product_idx):

        demand = int(
            inventory_snapshot.loc[
                idx,
                "Weekly_Demand"
            ]
        )

        # Beginning inventory
        inventory_snapshot.loc[
            idx,
            "Beginning_Inventory"
        ] = current_inventory

        # Demand consumption
        ending_inventory = max(
            current_inventory - demand,
            0
        )

        inventory_snapshot.loc[
            idx,
            "Ending_Inventory"
        ] = ending_inventory

        replenishment_qty = 0

        # Review every 4 weeks
        if (i + 1) % 4 == 0:

            target_inventory = int(
                inventory_snapshot.loc[
                    idx,
                    "Target_Inventory"
                ]
            )

            replenishment_qty = max(
                target_inventory - ending_inventory,
                0
            )

            inventory_snapshot.loc[
                idx,
                "Replenishment_Qty"
            ] = replenishment_qty

            current_inventory = (
                ending_inventory
                + replenishment_qty
            )

        else:

            current_inventory = ending_inventory

In [130]:
inventory_snapshot[
    inventory_snapshot["Product Card Id"]
    ==
    inventory_snapshot["Product Card Id"].iloc[0]
][
    [
        "Order_Date",
        "Weekly_Demand",
        "Beginning_Inventory",
        "Ending_Inventory",
        "Replenishment_Qty"
    ]
].head(12)

,Order_Date,Weekly_Demand,Beginning_Inventory,Ending_Inventory,Replenishment_Qty
0,2017-04-30,2,48.0,46.0,0
1,2017-05-07,3,46.0,43.0,0
2,2017-05-14,1,43.0,42.0,0
3,2017-05-21,3,42.0,39.0,0
4,2017-05-28,6,39.0,33.0,0
5,2017-06-04,3,33.0,30.0,0
6,2017-06-11,2,30.0,28.0,0
7,2017-06-18,2,28.0,26.0,0
8,2017-06-25,2,26.0,24.0,0
9,2017-07-02,7,24.0,17.0,0


In [ ]:
#inventory_snapshot.to_csv(
#    "../data/processed/inventory_snapshot_weekly.csv",
#    index=False)